## Installs and Imports

In [ ]:
!pip install -U torch torchvision
!pip install transformers datasets tqdm pandas scipy

In [ ]:
!pip install --force-reinstall --no-cache-dir scipy # Only needed within runpod environment
!pip uninstall -y Pillow
!pip install Pillow
!pip install numpy==1.26.4

In [ ]:
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import ViTForImageClassification, DeiTImageProcessor
from datasets import load_from_disk, load_dataset
import numpy as np
import copy

## Data Prep

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
processor = DeiTImageProcessor.from_pretrained('facebook/deit-tiny-patch16-224')

In [ ]:
dataset = load_dataset('tanganke/"tanganke/sun397', cache_dir="/workspace/.hf_cache") # https://huggingface.co/datasets/tanganke/sun397

split = dataset["train"].train_test_split(test_size=0.2, seed=66)

train = split["train"]
val = split["test"]
test = dataset["test"]

In [ ]:
def transform_example(batch):
    images = batch["image"]
    batch["pixel_values"] = processor(images, return_tensors="np")["pixel_values"]
    return batch

train = train.map(transform_example, batched=True, batch_size=64)
val = val.map(transform_example, batched=True, batch_size=64)
test = test.map(transform_example, batched=True, batch_size=64)

In [ ]:
train.save_to_disk('/workspace/preprocessed/SUN397/train_processed')
val.save_to_disk('/workspace/preprocessed/SUN397/val_processed')
test.save_to_disk('/workspace/preprocessed/SUN397/test_processed')

In [ ]:
train = load_from_disk('/workspace/preprocessed/SUN397/train_processed')
val = load_from_disk('/workspace/preprocessed/SUN397/val_processed')
test = load_from_disk('/workspace/preprocessed/SUN397/test_processed')

In [ ]:
train.set_format(type='torch', columns=["image", "label", "pixel_values"])
val.set_format(type='torch', columns=["image", "label", "pixel_values"])
test.set_format(type='torch', columns=["image", "label", "pixel_values"])

def collate_fn(batch):
    images = torch.stack([example["pixel_values"] for example in batch])
    labels = torch.tensor([example["label"] for example in batch])
    
    return {
        "pixel_values": images,
        "labels": labels
    }

train_loader = DataLoader(train, batch_size=8, shuffle=True, collate_fn=collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)
val_loader = DataLoader(val, batch_size=8, shuffle=False, collate_fn=collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)
test_loader = DataLoader(test, batch_size=8, shuffle=False, collate_fn=collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)

## Class Prep

In [ ]:
# https://github.com/huggingface/transformers/blob/main/src/transformers/models/deit/image_processing_deit.py
class Augmented(torch.nn.Module):
    def __init__(self, model, classifier=None, W=None, transform_stage=-1):
        super().__init__()
        self.model = model
        self.classifier = classifier if classifier is not None else torch.nn.Linear(self.model.config.hidden_size, 397)
        self.W = torch.from_numpy(W.astype(np.float32)).to(device) if W is not None else None
        self.transform_stage = transform_stage
    
    def forward(self, images):
        hidden_states = self.model.vit.embeddings(images)

        for i, layer_module in enumerate(self.model.vit.encoder.layer):
            layer_outputs = layer_module(hidden_states)
            hidden_states = layer_outputs[0]
            if i == self.transform_stage:
                if self.W is None:
                    self.W = torch.eye(hidden_states.shape[-1], device=hidden_states.device, dtype=hidden_states.dtype)
                cls = hidden_states[:, 0, :]
                cls = cls @ self.W
                hidden_states[:, 0, :] = cls
                break
            cls = hidden_states[:, 0, :]
        
        hidden_states = self.model.vit.layernorm(hidden_states)
        logits = self.classifier(hidden_states[:, 0, :])

        return logits, cls

In [ ]:
#  https://huggingface.co/facebook/deit-tiny-patch16-224
refer = ViTForImageClassification.from_pretrained("facebook/deit-tiny-patch16-224")

base = Augmented(copy.deepcopy(refer)).to(device)

## Fine-Tune Prep

In [ ]:
optimizer = torch.optim.AdamW([
    {'params': base.model.parameters(), 'lr': 1e-5},
    {'params': base.classifier.parameters(), 'lr': 1e-3}
], weight_decay=0.01)

criterion = torch.nn.CrossEntropyLoss()

EPOCHS = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=len(train_loader) * EPOCHS)

In [ ]:
best_val_loss = float('inf')
best_epoch = -1

for epoch in range(EPOCHS):
  print(f"Epoch {epoch}/{EPOCHS} - Best Val Loss: {best_val_loss:.4f}, (Epoch {best_epoch})")

  base.train()
  total_train_loss = 0
  train_steps = 0

  for batch in tqdm(train_loader, desc="Training"):
    optimizer.zero_grad()

    images = batch["pixel_values"].to(device, non_blocking=True)
    labels = batch["labels"].to(device, non_blocking=True)

    logits, _ = base(images)
    loss = criterion(logits, labels)

    loss.backward()
    optimizer.step()

    total_train_loss += loss.item()
    train_steps += 1

  avg_train_loss = total_train_loss / train_steps

  # Validation
  base.eval()
  correct = 0
  total = 0
  total_val_loss = 0
  val_steps = 0

  with torch.no_grad():
    for batch in tqdm(val_loader, desc="Validation"):
      images = batch["pixel_values"].to(device, non_blocking=True)
      labels = batch["labels"].to(device, non_blocking=True)

      logits, _ = base(images)
      loss = criterion(logits, labels)

      preds = torch.argmax(logits, dim=1)
      correct += (preds == labels).sum().item()
      total += labels.size(0)

      total_val_loss += loss.item()
      val_steps += 1

  avg_val_loss = total_val_loss / val_steps
  val_acc = correct / total

  print(f"[Epoch {epoch}] Train Loss: {avg_train_loss:.4f} | Validation Loss: {avg_val_loss:.4f} | Validation Accuracy: {val_acc:.4f}")

  if avg_val_loss < best_val_loss:
    best_val_loss = avg_val_loss
    best_epoch = epoch
    torch.save(base.state_dict(), "best_DeiT_SUN397.pt")

  scheduler.step()

In [ ]:
base = Augmented(copy.deepcopy(refer)).to(device).eval()

f_t = Augmented(copy.deepcopy(refer))
f_t.load_state_dict(torch.load("best_DeiT_SUN397.pt", map_location=device))
fine_tuned = Augmented(f_t.model, f_t.classifier).to(device).eval()

In [ ]:
correct_base = 0
correct_fine_tuned = 0
total = 0

base_loss = 0
fine_tuned_loss = 0
total_loss = 0

for batch in tqdm(test_loader, desc="Evaluating"):
    images = batch["pixel_values"].to(device, non_blocking=True)
    labels = batch["labels"].to(device, non_blocking=True)
    total += labels.size(0)

    logits_base, _ = base(images)
    base_loss += criterion(logits_base, labels).item()
    logits_fine_tuned, _ = fine_tuned(images)
    fine_tuned_loss += criterion(logits_fine_tuned, labels).item()
    total_loss += 1

    pred = logits_base.argmax(dim=1)
    correct_base += (pred == labels).sum().item()

    pred = logits_fine_tuned.argmax(dim=1)
    correct_fine_tuned += (pred == labels).sum().item()

In [ ]:
avg_base_loss = base_loss / total_loss
avg_best_loss = fine_tuned_loss / total_loss

accuracy_base = correct_base / total
accuracy_best = correct_fine_tuned / total
print(f"\nAverage base loss: {avg_base_loss:.4f}, Base Accuracy: {accuracy_base:.4f}")
print(f"Average best loss: {avg_best_loss:.4f}, Best Accuracy: {accuracy_best:.4f}")

In [ ]:
#  find /workspace -mindepth 1 -exec rm -rf {} +